![erd](./images/ERD-week5-exercise.png)

## Instruction

1. ใช้ project myshop เดิมจากของ WEEK 4 มาทำต่อได้เลยครับ (ใครที่หา project ของ WEEK 4 ไม่เจอแล้ว ให้กลับไปทำตามขั้นตอนใน week4_exercises.ipynb)
2. ใช้ database `shop` เดิมได้เลยเช่นกันครับ

**หมายเหตุ: ถ้านักศึกษาใช้ database เดิมจาก WEEK4 ซึ่งนักศึกษาเคยเพิ่มหรือแก้ไขข้อมูลบางส่วนแล้ว ดังนั้น output อาจจะไม่ตรงกับในตัวอย่างทั้งหมดไม่ต้องตกใจนะครับ**

In [1]:
import os
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = "true"

# import modules
from shop.models import *

### 1. annotate(), F()

1.1 ให้นักศึกษา query ข้อมูล `Payment` โดยให้เพิ่ม field ราคาที่ลบกับส่วนลดแล้ว โดยกำหนดให้ชื่อ field ว่า "after_discount_price" โดยใช้แสดงข้อมูล 10 ตัวแรกเรียงตาม "after_discount_price" จากมากไปน้อย (0.25 คะแนน)

**หมายเหตุ: จะต้องใช้ annotate() นะครับ ให้เอา `Payment.price` - `Payment.discount`**

ตัวอย่าง Output

```
ID: 92, PRICE: 1200500.00, DISCOUNT 29433.25, AFTER_DISCOUNT 1171066.75
ID: 82, PRICE: 1200280.00, DISCOUNT 46229.40, AFTER_DISCOUNT 1154050.60
ID: 137, PRICE: 1200690.00, DISCOUNT 71407.25, AFTER_DISCOUNT 1129282.75
ID: 105, PRICE: 1200390.00, DISCOUNT 105019.11, AFTER_DISCOUNT 1095370.89
ID: 45, PRICE: 1218900.00, DISCOUNT 126859.95, AFTER_DISCOUNT 1092040.05
ID: 7, PRICE: 1201200.00, DISCOUNT 113446.20, AFTER_DISCOUNT 1087753.80
ID: 18, PRICE: 1202190.00, DISCOUNT 121922.64, AFTER_DISCOUNT 1080267.36
ID: 77, PRICE: 379000.00, DISCOUNT 19397.00, AFTER_DISCOUNT 359603.00
ID: 127, PRICE: 320450.00, DISCOUNT 14578.90, AFTER_DISCOUNT 305871.10
ID: 125, PRICE: 320399.00, DISCOUNT 17939.55, AFTER_DISCOUNT 302459.45
```

In [2]:
from django.db.models import Count, F, Value, Q
from django.db.models.functions import Length, Upper, Concat
from django.db.models.lookups import GreaterThan

In [3]:
p1 = Payment.objects.annotate(after_discount_price=F("price") - F("discount"))
for i in p1.order_by("-after_discount_price")[:10]:
    print(f"ID: {i.order_id}, PRICE: {i.price}, DISCOUNT: {i.discount}, AFTER_DISCOUNT: {i.after_discount_price}")

ID: 90, PRICE: 1200500.00, DISCOUNT: 29433.25, AFTER_DISCOUNT: 1171066.75
ID: 80, PRICE: 1200280.00, DISCOUNT: 46229.40, AFTER_DISCOUNT: 1154050.60
ID: 135, PRICE: 1200690.00, DISCOUNT: 71407.25, AFTER_DISCOUNT: 1129282.75
ID: 103, PRICE: 1200390.00, DISCOUNT: 105019.11, AFTER_DISCOUNT: 1095370.89
ID: 43, PRICE: 1218900.00, DISCOUNT: 126859.95, AFTER_DISCOUNT: 1092040.05
ID: 5, PRICE: 1201200.00, DISCOUNT: 113446.20, AFTER_DISCOUNT: 1087753.80
ID: 16, PRICE: 1202190.00, DISCOUNT: 121922.64, AFTER_DISCOUNT: 1080267.36
ID: 75, PRICE: 379000.00, DISCOUNT: 19397.00, AFTER_DISCOUNT: 359603.00
ID: 125, PRICE: 320450.00, DISCOUNT: 14578.90, AFTER_DISCOUNT: 305871.10
ID: 123, PRICE: 320399.00, DISCOUNT: 17939.55, AFTER_DISCOUNT: 302459.45


In [4]:
p2 = p1.filter(after_discount_price__gt=500000)
for i in p2:
    print(f"ID: {i.order_id}, PRICE: {i.price}, DISCOUNT: {i.discount}, AFTER_DISCOUNT: {i.after_discount_price}")

ID: 5, PRICE: 1201200.00, DISCOUNT: 113446.20, AFTER_DISCOUNT: 1087753.80
ID: 16, PRICE: 1202190.00, DISCOUNT: 121922.64, AFTER_DISCOUNT: 1080267.36
ID: 43, PRICE: 1218900.00, DISCOUNT: 126859.95, AFTER_DISCOUNT: 1092040.05
ID: 80, PRICE: 1200280.00, DISCOUNT: 46229.40, AFTER_DISCOUNT: 1154050.60
ID: 90, PRICE: 1200500.00, DISCOUNT: 29433.25, AFTER_DISCOUNT: 1171066.75
ID: 103, PRICE: 1200390.00, DISCOUNT: 105019.11, AFTER_DISCOUNT: 1095370.89
ID: 135, PRICE: 1200690.00, DISCOUNT: 71407.25, AFTER_DISCOUNT: 1129282.75


In [ ]:
# 1.3 ให้นักศึกษาเรียงลำดับข้อมูลลูกค้า (`Customer`) โดยเรียงลำดับตามลำดับตัวอักษร `น้อยไปมาก` (A-Z) จากชื่อเต็มของลูกค้า (`full_name`) โดยแสดง 5 คนแรก (0.5 คะแนน)

**Hint:** Field `full_name` นั้นจะต้องถูก annotate ขึ้นมาโดยการนำ `first_name` มาต่อกับ `last_name` โดยใช้ `Concat(*expressions, **extra)` 

**Hint:** แปลง object เป็น dict ใช้ `values()` [doc](https://docs.djangoproject.com/en/5.0/ref/models/querysets/#values)

```python
>>> Blog.objects.filter(name__startswith="Beatles").values()
<QuerySet [{'id': 1, 'name': 'Beatles Blog', 'tagline': 'All the latest Beatles news.'}]>
```

**Hint:** อยาก print dictionary สวยๆ ใช้ `json.dumps`

```python
print(json.dumps(dictionary, indent=4, sort_keys=False))
```

[doc](https://docs.djangoproject.com/en/5.0/ref/models/database-functions/#concat)

ตัวอย่าง Output 

```python
{
    "id": 17,
    "email": "anantaya.deena@gmail.com",
    "address": {
        "district": "Yan Nawa",
        "location": "60 Thanon Chan Road",
        "province": "Bangkok",
        "postal_code": 10120
    },
    "full_name": "Anantaya Tontong"
}
{
    "id": 25,
    "email": "bancha.zeyou@gmail.com",
    "address": {
        "district": "Watthana",
        "location": "6 Thong Lo Road",
        "province": "Bangkok",
        "postal_code": 10110
    },
    "full_name": "Bancha Kittisompong"
}
{
    "id": 19,
    "email": "chayapol.231@gmail.com",
    "address": {
        "district": "Hang Chat",
        "location": "160 Lampang Road",
        "province": "Lampang",
        "postal_code": 52190
    },
    "full_name": "Chayapol Komprach"
}
{
    "id": 4,
    "email": "dejwit.tt@gmail.com",
    "address": {
        "district": "Chiang Khan",
        "location": "150 Loei Road",
        "province": "Loei",
        "postal_code": 42110
    },
    "full_name": "Dejwit Tangjareonsakul"
}
{
    "id": 11,
    "email": "jack.maa@gmail.com",
    "address": {
        "district": "Bang Khen",
        "location": "88 Phahonyothin Road",
        "province": "Bangkok",
        "postal_code": 10220
    },
    "full_name": "Jack Maa"
}

```

In [5]:
import json
c1 = Customer.objects.annotate(
    fullname = Concat("first_name", Value(" "), "last_name")
).order_by("fullname")[:5].values("id", "email", "address", "fullname")

for i in c1:
    print(json.dumps(i, indent=4, sort_keys=False))

{
    "id": 17,
    "email": "anantaya.deena@gmail.com",
    "address": {
        "district": "Yan Nawa",
        "location": "60 Thanon Chan Road",
        "province": "Bangkok",
        "postal_code": 10120
    },
    "fullname": "Anantaya Tontong"
}
{
    "id": 25,
    "email": "bancha.zeyou@gmail.com",
    "address": {
        "district": "Watthana",
        "location": "6 Thong Lo Road",
        "province": "Bangkok",
        "postal_code": 10110
    },
    "fullname": "Bancha Kittisompong"
}
{
    "id": 19,
    "email": "chayapol.231@gmail.com",
    "address": {
        "district": "Hang Chat",
        "location": "160 Lampang Road",
        "province": "Lampang",
        "postal_code": 52190
    },
    "fullname": "Chayapol Komprach"
}
{
    "id": 4,
    "email": "dejwit.tt@gmail.com",
    "address": {
        "district": "Chiang Khan",
        "location": "150 Loei Road",
        "province": "Loei",
        "postal_code": 42110
    },
    "fullname": "Dejwit Tangjareonsakul"
}


### 3. aggregation - count(), sum(), avg()


3.1 ให้นักศึกษาหาค่าเฉลี่ยของราคาสินค้า (`Product.price`) ที่มีจำนวนคงเหลือ (`Product.remaining_amount`) มากกว่า 0 (0.2 คะแนน)

``` PYTHON
{'avg': Decimal('29308.000000000000')}
```


In [6]:
Product.objects.filter(remaining_amount__gt=0).aggregate(
    avg = Avg("price", default=0)
)

{'avg': Decimal('29308.000000000000')}

3.2 จงหาผลรวมราคา (`CartItem.product.price`) ที่อยู่ในตระกร้าสินค้าของเดือน `พฤษภาคม` (ดูจาก `Cart.create_date`) (0.2 คะแนน)

```PYTHON
{'sum': Decimal('9912555.00')}

```

In [7]:
from django.db.models import Sum
cc = CartItem.objects.filter(cart__create_date__month=5).aggregate(
    sum = Sum("product__price")
)

cc

{'sum': Decimal('9912555.00')}

3.3 ให้นักศึกษานับจำนวนสินค้าที่อยู่ประเภท `Electronics`,  `Jewelry` และ ราคาของสินค้าอยู่ในช่วง 8,000.00 - 50,000.00 (0.2 คะแนน)

```
PRODUCT CATEGORY NAME: Electronics, PRODUCT COUNT: 6
PRODUCT CATEGORY NAME: Jewelry, PRODUCT COUNT: 1
```

In [70]:
p41 = Product.objects.filter(categories__name="Electronics").filter(price__gte=8000, price__lte=50000).count()
p42 = Product.objects.filter(categories__name="Jewelry").filter(price__gte=8000, price__lte=50000).count()
print(f"PRODUCT CATEGORY NAME: Electronics, PRODUCT COUNT: {p41}")
print(f"PRODUCT CATEGORY NAME: Jewelry, PRODUCT COUNT: {p42}")

PRODUCT CATEGORY NAME: Electronics, PRODUCT COUNT: 6
PRODUCT CATEGORY NAME: Jewelry, PRODUCT COUNT: 1


3.4 ให้นักศึกษามีการใช้ `Subquery` เพื่อหาคำสั่งซื้อ (`Order`) ล่าสุดของลูกค้า (`Customer`) แต่ละคน โดยมีการเรียงลำดับคำสั่งซื้อตามวันที่ที่สั่งซื้อ และแสดงผลข้อมูลตามตัวอย่างที่กำหนดไว้ (0.4 คะแนน)
```txt
# ตัวอย่างผลลัพธ์บางส่วน
CUSTOMER NAME: Sek Loso, ORDER DATE: 2024-05-13
CUSTOMER NAME: Warit Pititat, ORDER DATE: 2024-05-26
CUSTOMER NAME: Chayapol Komprach, ORDER DATE: 2024-05-26
CUSTOMER NAME: Kwantip Aekopas, ORDER DATE: 2024-05-28
CUSTOMER NAME: Kerikwan Laopong, ORDER DATE: 2024-05-30
CUSTOMER NAME: Pattanapoom Srisawangjan, ORDER DATE: 2024-05-30
CUSTOMER NAME: Manit Senapan, ORDER DATE: 2024-06-06
CUSTOMER NAME: Techathorn Sriwanwilai, ORDER DATE: 2024-06-07
CUSTOMER NAME: Anantaya Tontong, ORDER DATE: 2024-06-08
CUSTOMER NAME: Kittitorn Wilairat, ORDER DATE: 2024-06-11
...
```

In [71]:
from django.db.models import OuterRef, Subquery

In [23]:
newest = Order.objects.filter(customer=OuterRef("pk")).order_by("-order_date")
c10 = Customer.objects.annotate(nw=Subquery(newest.values("order_date")[:1])).order_by("nw")

for i in c10:
    print(f"CUSTOMER NAME: {i.first_name} {i.last_name}, ORDER DATE: {i.nw}")

CUSTOMER NAME: Sek Loso, ORDER DATE: 2024-05-13
CUSTOMER NAME: Warit Pititat, ORDER DATE: 2024-05-26
CUSTOMER NAME: Chayapol Komprach, ORDER DATE: 2024-05-26
CUSTOMER NAME: Kwantip Aekopas, ORDER DATE: 2024-05-28
CUSTOMER NAME: Kerikwan Laopong, ORDER DATE: 2024-05-30
CUSTOMER NAME: Pattanapoom Srisawangjan, ORDER DATE: 2024-05-30
CUSTOMER NAME: Manit Senapan, ORDER DATE: 2024-06-06
CUSTOMER NAME: Techathorn Sriwanwilai, ORDER DATE: 2024-06-07
CUSTOMER NAME: Anantaya Tontong, ORDER DATE: 2024-06-08
CUSTOMER NAME: Kittitorn Wilairat, ORDER DATE: 2024-06-11
CUSTOMER NAME: Porntipa Pasakul, ORDER DATE: 2024-06-12
CUSTOMER NAME: Peeraya Wichienchan, ORDER DATE: 2024-06-13
CUSTOMER NAME: Rachanon Hongpatra, ORDER DATE: 2024-06-14
CUSTOMER NAME: Suwichada Jarastham, ORDER DATE: 2024-06-14
CUSTOMER NAME: Pong Sawadiwong, ORDER DATE: 2024-06-15
CUSTOMER NAME: Dejwit Tangjareonsakul, ORDER DATE: 2024-06-16
CUSTOMER NAME: Bancha Kittisompong, ORDER DATE: 2024-06-17
CUSTOMER NAME: Yanisa Nantapak

### 4. one-to-one & one-to-many

4.1 จงสร้าง Order และ Payment ของลูกค้า Sek Loso จำนวน 2 รายการ โดยแต่ละ Order มีสินค้า 1 รายการ และชำระเงินด้วย Payment คนละรายการ จากนั้น เขียน query แสดงข้อมูล Order และ Payment ทั้งหมดของ Sek Loso (0.25 คะแนน)
```txt
# ตัวอย่างผลลัพธ์บางส่วน
CUSTOMER: Sek Loso, ORDER ID: 48, PRICE: 1699.00, DISCOUNT: 112.27, PRODUCT: Bird Cage
CUSTOMER: Sek Loso, ORDER ID: 59, PRICE: 1740.00, DISCOUNT: 140.81, PRODUCT: Microwave Oven
```

In [27]:
from datetime import datetime, date, time, timedelta

In [28]:
cus = Customer.objects.get(first_name="Sek", last_name="Loso")
sek_order = Order.objects.create(customer=cus, order_date=date(2023, 1, 1))
sek_product = Product.objects.get(name="Laptop")
sek_pay = Payment.objects.create(order=sek_order, payment_date=date(2023, 1, 2), price=1000, discount=20)
sek_orderitem = OrderItem.objects.create(order=sek_order, product=sek_product, amount=1)
sek_payment = PaymentItem.objects.create(payment=sek_pay, order_item=sek_orderitem, price=1000, discount=20)

sek_order2 = Order.objects.create(customer=cus, order_date=date(2023, 2, 1))
sek_product2 = Product.objects.get(name="Smartphone")
sek_pay2 = Payment.objects.create(order=sek_order2, payment_date=date(2023, 2, 2), price=1040, discount=20)
sek_orderitem2 = OrderItem.objects.create(order=sek_order2, product=sek_product2, amount=1)
sek_payment2 = PaymentItem.objects.create(payment=sek_pay2, order_item=sek_orderitem2, price=1040, discount=20)

In [53]:
sek1_order = OrderItem.objects.filter(order__customer__first_name="Sek")
for i in sek1_order:
    print(f"CUSTOMER: {i.order.customer.first_name} {i.order.customer.last_name}, \
    ORDER ID: {i.id}, PRICE: {i.order.payment.price}, DISCOUNT: {i.order.payment.discount}, PRODUCT: {i.product.name}")
#     orderitem = OrderItem.objects.filter(order=i)
#     for j in orderitem:
#         print(f"{j.product.name}", end="")
#     print()

CUSTOMER: Sek Loso, ORDER ID: 95, PRICE: 1699.00, DISCOUNT: 112.27, PRODUCT: Bird Cage
CUSTOMER: Sek Loso, ORDER ID: 96, PRICE: 1699.00, DISCOUNT: 112.27, PRODUCT: Hot Wheels Track Builder System Race Crate
CUSTOMER: Sek Loso, ORDER ID: 117, PRICE: 1740.00, DISCOUNT: 140.81, PRODUCT: Microwave Oven
CUSTOMER: Sek Loso, ORDER ID: 118, PRICE: 1740.00, DISCOUNT: 140.81, PRODUCT: Men's Polo Shirt
CUSTOMER: Sek Loso, ORDER ID: 367, PRICE: 1000.00, DISCOUNT: 20.00, PRODUCT: Laptop
CUSTOMER: Sek Loso, ORDER ID: 368, PRICE: 1040.00, DISCOUNT: 20.00, PRODUCT: Smartphone


4.2 ให้นักศึกษาดึงข้อมูล `Payment` ที่มี `PaymentMethod` ที่เป็น 'CREDIT' และมีราคาหลังหักส่วนลด (`Payment.price` - `Payment.discount`) ต่ำกว่า 500 และให้แสดงจำนวนของ `PaymentItem` ของแต่ละรายการ Payment (0.25 คะแนน)
```txt
# ตัวอย่างผลลัพธ์
PAYMENT ID: 40, AFTER_DISCOUNT: 493.06, ITEM COUNT: 2
PAYMENT ID: 13, AFTER_DISCOUNT: 448.54, ITEM COUNT: 2
```

In [66]:
hey = Payment.objects.filter(paymentmethod__method='CREDIT'
                                  ).annotate(price_after=F("price")-F("discount")
                                            ).filter(price_after__lt=500)
for i in hey:
#     heyhey = PaymentItem.objects.filter(payment_id=i.id).count()
    print(F"PAYMENT ID: {i.id}, AFTER_DISCOUNT: {i.price_after},\
    ITEM COUNT: {PaymentItem.objects.filter(payment_id=i.id).count()}")

PAYMENT ID: 13, AFTER_DISCOUNT: 448.54, ITEM COUNT: 2
PAYMENT ID: 40, AFTER_DISCOUNT: 493.06, ITEM COUNT: 2


### 5. many-to-many

5.1 ให้นักศึกษาค้นหาข้อมูลสินค้า (`Product`) ที่อยู่ในทั้งหมวดหมู่สินค้า "Information Technology" และ "Electronics" 10 รายการแรก (เรียงลำดับด้วย `Product.id`) และแสดงชื่อประเภทสินค้า (`ProductCategory`) (0.25 คะแนน)

ตัวอย่าง Output บางส่วน

```
PRODUCT ID: 1, PRODUCT NAME: Smartphone, PRODUCT CATEGORY: Information technology, Electronics
PRODUCT ID: 2, PRODUCT NAME: Laptop, PRODUCT CATEGORY: Information technology, Electronics
PRODUCT ID: 3, PRODUCT NAME: Smart TV, PRODUCT CATEGORY: Information technology, Electronics
```

In [103]:
prod = Product.objects.filter(categories__name='Information Technology'
                             ).filter(categories__name='Electronics and Toys').order_by('id')[:10]
for i in prod:
    print(f"PRODUCT ID: {i.id}, PRODUCT NAME: {i.name}, PRODUCT CATEGORY: ", end="")
    for j in i.categories.all():
        print(f"{j.name}", end=", ")
    print()

PRODUCT ID: 1, PRODUCT NAME: Smartphone, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 2, PRODUCT NAME: Laptop, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 3, PRODUCT NAME: Smart TV, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 4, PRODUCT NAME: Bluetooth Earphones, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 5, PRODUCT NAME: Tablet, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 6, PRODUCT NAME: Gaming Console, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 7, PRODUCT NAME: Digital Camera, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 8, PRODUCT NAME: Wireless Router, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 9, PRODUCT NAME: Portable Power Bank, PRODUCT CATEGORY: Information Technology, Electronics and Toys, 
PRODUCT ID: 10, PRODUCT N

5.2 ให้นักศึกษาทำตามขั้นตอนดังนี้ (0.25 คะแนน)

1. เปลี่ยนชื่อหมวดหมู่สินค้า `Electronics` เป็น `Electronics and Toys` แทน
2. ย้ายสินค้าที่อยู่ใน `Toys and Games` ไปอยู่ในหมวดหมู่ `Electronics and Toys` และ ลบหมวดหมู่สินค้า `Toys and Games` 
3. ค้นหาว่าสินค้าที่มีประเภทสินค้าเป็น `Electronics and Toys` ทั้งหมดมีจำนวนเท่าไหร่

In [110]:
elec = ProductCategory.objects.get(name='Electronics')
elec.name = 'Electronics and Toys'
elec.save()

DoesNotExist: ProductCategory matching query does not exist.

In [111]:
toy = Product.objects.filter(categories__name='Toys and Games')
for i in toy:
    i.categories.add(elec)

In [112]:
ProductCategory.objects.filter(name='Toys and Games').delete()

(0, {})

In [113]:
ProductCategory.objects.filter(name='Electronics and Toys')

<QuerySet [<ProductCategory: ProductCategory object (2)>]>